In [21]:
import pandas as pd
import os
import sys
import plotly.express as px
import warnings

# Ignorar avisos irrelevantes do pandas
warnings.filterwarnings('ignore')

# Define a pasta local onde estão os seus CSVs
pasta_dados = 'data/'
dfs = []

print("Iniciando a leitura dos datasets locais...")

# Lista todos os arquivos terminados em .csv dentro da pasta 'data'
arquivos_csv = [f for f in os.listdir(pasta_dados) if f.endswith('.csv')]

for arquivo in arquivos_csv:
    caminho_completo = os.path.join(pasta_dados, arquivo)
    print(f"Carregando {arquivo}...")
    try:
        # Lê o arquivo direto do seu HD (D:) usando as mesmas regras de formatação
        df = pd.read_csv(caminho_completo, sep=';', encoding='latin1', on_bad_lines='skip', low_memory=False)
        dfs.append(df)
    except Exception as e:
        print(f" -> Erro ao ler o arquivo {arquivo}: {e}")

# Verificação de segurança (Parada se a pasta estiver vazia ou com erro)
if not dfs:
    print("ERRO CRÍTICO: Nenhum dataset foi carregado. Verifique se os arquivos estão na pasta 'data'. Execução interrompida.")
    sys.exit()

# Concatenação de todos os anos
df_all = pd.concat(dfs, ignore_index=True)
print(f"\nLeitura concluída com sucesso! Volume total da base: {len(df_all)} registros.")

Iniciando a leitura dos datasets locais...
Carregando emlurb_2025.csv...
Carregando resources_031c3ad3-265f-4d9c-830a-7d1f85f830fa_2024-central-de-atendimento-de-servicos-da-emlurb-156.csv...
Carregando resources_192dc99d-b248-4671-8b3d-a8fe7210bf3c_2021-central-de-atendimento-de-servicos-da-emlurb-156.csv...
Carregando resources_30ba6cc0-17a8-4f38-8c86-55184c87ead1_2023-central-de-atendimento-de-servicos-da-emlurb-156.csv...
Carregando resources_6c7fceed-3495-449c-bbd6-01bcd3e760bd_2022-central-de-atendimento-de-servicos-da-emlurb-156.csv...
Carregando resources_aba3d1a0-c456-4a3b-91e1-4aaf30704717_2020-central-de-atendimento-de-servicos-da-emlurb-156.csv...

Leitura concluída com sucesso! Volume total da base: 572826 registros.


In [22]:
print("Iniciando rotina de limpeza e engenharia de dados...")

# 1. Tratamento de Datas
df_all['DATA_DEMANDA'] = pd.to_datetime(df_all['DATA_DEMANDA'], format='mixed', errors='coerce')
df_all['DATA_ULT_SITUACAO'] = pd.to_datetime(df_all['DATA_ULT_SITUACAO'], format='mixed', errors='coerce')

# 2. Remoção Estrita de Nulos
reg_antes = len(df_all)
df_all.dropna(subset=['DATA_DEMANDA'], inplace=True)
print(f"Linhas descartadas por falta de DATA_DEMANDA: {reg_antes - len(df_all)}")

# 3. Engenharia de Variáveis (Features) para EDA
df_all['Ano_Mes'] = df_all['DATA_DEMANDA'].dt.to_period('M').astype(str)
df_all['Mes'] = df_all['DATA_DEMANDA'].dt.month
# Extraindo o nome do dia da semana
df_all['Dia_Semana'] = df_all['DATA_DEMANDA'].dt.day_name()

# 4. Padronização de Strings (Tirar espaços invisíveis e deixar tudo maiúsculo)
df_all['BAIRRO'] = df_all['BAIRRO'].astype(str).str.strip().str.upper()
df_all['GRUPOSERVICO_DESCRICAO'] = df_all['GRUPOSERVICO_DESCRICAO'].astype(str).str.strip().str.upper()

print("Limpeza finalizada. A base está pronta para a Análise Exploratória.")

Iniciando rotina de limpeza e engenharia de dados...
Linhas descartadas por falta de DATA_DEMANDA: 0
Limpeza finalizada. A base está pronta para a Análise Exploratória.


In [23]:
vol_mensal = df_all['Ano_Mes'].value_counts().sort_index().reset_index()
vol_mensal.columns = ['Mês', 'Volume de Denúncias']

fig1 = px.line(vol_mensal, x='Mês', y='Volume de Denúncias', 
               title='1. Evolução do Volume Total de Denúncias (Série Histórica)',
               markers=True)
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()

In [24]:
top5_cat = df_all['GRUPOSERVICO_DESCRICAO'].value_counts().head(5).index
df_top5_cat = df_all[df_all['GRUPOSERVICO_DESCRICAO'].isin(top5_cat)]

vol_mes_cat = df_top5_cat.groupby(['Mes', 'GRUPOSERVICO_DESCRICAO']).size().reset_index(name='Volume')

fig2 = px.bar(vol_mes_cat, x='Mes', y='Volume', color='GRUPOSERVICO_DESCRICAO',
              barmode='group', title='2. Sazonalidade das Categorias (Top 5 Serviços por Mês)',
              labels={'Mes': 'Mês do Ano (1 a 12)', 'GRUPOSERVICO_DESCRICAO': 'Categoria'})
fig2.show()

In [25]:
heatmap_data = df_all.groupby(['Dia_Semana', 'Mes']).size().reset_index(name='Volume')

# Ordem dos dias para ficar visualmente correto no eixo Y do Plotly
ordem_dias = ['Sunday', 'Saturday', 'Friday', 'Thursday', 'Wednesday', 'Tuesday', 'Monday']

fig3 = px.density_heatmap(
    heatmap_data, 
    x='Mes', 
    y='Dia_Semana', 
    z='Volume',
    title='3. Padrão de Acessos: Dias da Semana vs Meses do Ano',
    category_orders={'Dia_Semana': ordem_dias},
    color_continuous_scale='YlGnBu',
    labels={'Mes': 'Mês do Ano', 'Dia_Semana': 'Dia da Semana'}
)
fig3.show()

In [26]:
# Top 10 Bairros (Barras Horizontais)
top10_bairros = df_all['BAIRRO'].value_counts().head(10).reset_index()
top10_bairros.columns = ['Bairro', 'Volume']

fig4 = px.bar(top10_bairros, x='Volume', y='Bairro', orientation='h', 
              title='4. Ranking de Volume: Top 10 Bairros Críticos',
              color='Volume', color_continuous_scale='Reds')
fig4.update_layout(yaxis={'categoryorder':'total ascending'})
fig4.show()

# Top 20 Bairros (Treemap)
top20_bairros = df_all['BAIRRO'].value_counts().head(20).reset_index()
top20_bairros.columns = ['Bairro', 'Volume']

fig5 = px.treemap(top20_bairros, path=['Bairro'], values='Volume',
                  title='5. Representatividade Visual Espacial: Top 20 Bairros',
                  color='Volume', color_continuous_scale='Blues')
fig5.show()

In [27]:
top5_bairros_nomes = df_all['BAIRRO'].value_counts().head(5).index
df_bairros_criticos = df_all[df_all['BAIRRO'].isin(top5_bairros_nomes)]

# Calcular a proporção percentual
df_prop = df_bairros_criticos.groupby(['BAIRRO', 'SITUACAO']).size().reset_index(name='Contagem')
df_prop['Porcentagem'] = df_prop.groupby('BAIRRO')['Contagem'].transform(lambda x: x / x.sum() * 100)

fig6 = px.bar(
    df_prop, 
    x='BAIRRO', 
    y='Porcentagem', 
    color='SITUACAO',
    title='6. Proporção de Resoluções nos 5 Bairros Críticos (100% Empilhado)',
    labels={'Porcentagem': 'Porcentagem (%)', 'BAIRRO': 'Bairro'},
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig6.show()

In [28]:
top10_vias = df_all['LOGRADOURO'].value_counts().head(10).reset_index()
top10_vias.columns = ['Logradouro', 'Quantidade_Defeitos']


fig_vias = px.bar(
    top10_vias,
    x='Quantidade_Defeitos',
    y='Logradouro',
    orientation='h', 
    title='7. Top 10 Vias Mais Críticas (Maior Número de Defeitos)',
    labels={'Quantidade_Defeitos': 'Nº de Ocorrências', 'Logradouro': 'Via/Logradouro'},
    color='Quantidade_Defeitos', 
    color_continuous_scale='Reds' 
)


fig_vias.update_layout(yaxis={'categoryorder':'total ascending'})

fig_vias.show()

In [29]:
status_balanco = df_all['SITUACAO'].value_counts().reset_index()
status_balanco.columns = ['Situacao', 'Total']


fig8 = px.pie(
    status_balanco, 
    values='Total', 
    names='Situacao', 
    hole=0.5, 
    title='8. Balanço de Eficiência Pública (Gráfico de Rosca)',
    color_discrete_sequence=px.colors.qualitative.Pastel, 
    labels={'Situacao': 'Status', 'Total': 'Chamados'}
)


fig8.update_traces(textinfo='percent+label', pull=[0.1 if c == 'PENDENTE' else 0 for c in status_balanco['Situacao']])

fig8.show()

In [30]:
# 1. Preparação dos dados: Isolar o maior grupo e contar sub-serviços
maior_grupo = df_all['GRUPOSERVICO_DESCRICAO'].value_counts().idxmax()
df_detalhe = df_all[df_all['GRUPOSERVICO_DESCRICAO'] == maior_grupo]
top10_servicos = df_detalhe['SERVICO_DESCRICAO'].value_counts().head(10).reset_index()
top10_servicos.columns = ['Serviço', 'Volume']

# 2. Gráfico de Barras Horizontais com gradiente de cor
fig9 = px.bar(
    top10_servicos, 
    x='Volume', 
    y='Serviço', 
    orientation='h',
    title=f'9. Detalhamento Crítico: Principais Queixas em "{maior_grupo}"',
    color='Volume',
    color_continuous_scale='Reds', # Tons de vermelho para indicar criticidade
    labels={'Volume': 'Qtd. Ocorrências', 'Serviço': 'Subcategoria de Serviço'}
)

# Ajuste para as barras ficarem em ordem decrescente
fig9.update_layout(yaxis={'categoryorder':'total ascending'}, showlegend=False)
fig9.show()

In [31]:
# 1. Preparação: Filtrar os Top 10 Bairros E as Top 10 Categorias
top_bairros = df_all['BAIRRO'].value_counts().head(10).index
top_categorias = df_all['GRUPOSERVICO_DESCRICAO'].value_counts().head(10).index

df_identidade = df_all[
    (df_all['BAIRRO'].isin(top_bairros)) & 
    (df_all['GRUPOSERVICO_DESCRICAO'].isin(top_categorias))
]

# 2. Criar matriz de cruzamento
matriz_identidade = pd.crosstab(df_identidade['BAIRRO'], df_identidade['GRUPOSERVICO_DESCRICAO'])

# 3. Gráfico de Calor (Heatmap) Ajustado
fig10 = px.imshow(
    matriz_identidade,
    labels=dict(x="Categoria do Problema", y="Bairro", color="Volume"),
    title='10. Identidade Urbana: Bairros vs. Principais Categorias',
    color_continuous_scale='Viridis',
    aspect="auto",
    height=600 # Aumenta a altura do gráfico para dar respiro
)

# Arrumando a bagunça: inclina as palavras e deixa no eixo de baixo para não bater no título
fig10.update_layout(
    xaxis_tickangle=-45,
    margin=dict(b=120) 
)

fig10.show()